# PD05 · Particionamiento y comparación pandas–Polars

Continuación de PD01–PD03 basada en las clases del 9 y 10 de octubre de 2026. Tiempo orientativo: 210 minutos, con el entorno y los datos preparados.

Copia esta plantilla a `soluciones/` conservando su nombre. Escribe allí el código y las interpretaciones. Las celdas iniciales muestran las lecturas; las actividades quedan a tu cargo. Antes de entregar, reinicia el kernel y ejecuta todas las celdas en orden.

Utiliza directamente las API de pandas, Polars y GeoPandas. Mantén visibles las conversiones, máscaras, agrupaciones, uniones y exportaciones. No uses `talleres.py`, `pqrs_talleres.py`, comandos de shell ni funciones que resuelvan varias etapas sin mostrar su implementación. Un bucle de medición es válido si su cuerpo contiene las operaciones que estás midiendo. No instales paquetes desde el notebook.

[Índice](../ejercicios/README.md) · [Pandas](../docs/manual-pandas.md) · [Polars](../docs/manual-polars.md) · [GeoPandas](../docs/manual-geopandas.md)

## Ubicación de trabajo

Abre Jupyter desde la raíz del repositorio con el entorno del curso activo. Cambia `REPO` por la ruta real de tu copia. Las rutas son explícitas y funcionan tanto desde `ejercicios/` como desde `soluciones/`. Si falta un archivo, revisa su ubicación; no sustituyas silenciosamente la entrada por otro dataset.

## Pregunta y preparación

¿Cómo cambian lectura, escritura y procesamiento al variar formato, particionamiento y motor?

Requiere el Parquet completo y la capa municipal del GeoPackage de PD04 y las dependencias del entorno del curso: `requirements.txt` y `requirements_pqrs.txt`. Trabaja en CPU. No dupliques filas para simular volumen. Esta actividad estudia los resultados de este equipo y estos datos.

Las lecturas siguientes son directas. Para comparar CSV más adelante, escribe explícitamente `pd.read_csv` y `pl.read_csv`, sus separadores, política de nulos y tipos. No uses inferencia diferente de tipos como si fuera trabajo equivalente.

El particionamiento será manual: un `groupby` divide la tabla y cada grupo se escribe en un archivo independiente. Los valores de las claves permanecen dentro de cada archivo. Un inventario CSV registra dónde está cada parte; no se configura un catálogo ni un motor de particiones.


In [1]:
import os
import time
import platform
import pandas as pd
import polars as pl
import geopandas as gpd
import matplotlib.pyplot as plt
from pathlib import Path

REPO = "/mnt/c/Users/TUPTC/bigdata/big-data-postgraduate"
INPUT = REPO + "/kit/salidas/soluciones/PD04/analisis_integrados.parquet"
OUT = REPO + "/kit/salidas/soluciones/PD05"
os.makedirs(OUT, exist_ok=True)
print("Entrada:", INPUT)
print("Salidas:", OUT)
print(platform.platform())
print("pandas:", pd.__version__, "Polars:", pl.__version__, "GeoPandas:", gpd.__version__)

Entrada: /mnt/c/Users/TUPTC/bigdata/big-data-postgraduate/kit/salidas/soluciones/PD04/analisis_integrados.parquet
Salidas: /mnt/c/Users/TUPTC/bigdata/big-data-postgraduate/kit/salidas/soluciones/PD05
Linux-6.18.54.1-microsoft-standard-WSL2-x86_64-with-glibc2.43
pandas: 2.2.3 Polars: 1.35.2 GeoPandas: 1.2.0


In [6]:
datos_pd = pd.read_parquet(INPUT, engine="pyarrow")
display(datos_pd.head())
print(datos_pd.shape)
print(datos_pd.dtypes)

,dpto_ccdgo,mpio_ccdgo,mpio_cdpmp,dpto_cnmbr,mpio_cnmbr,mpio_crslc,mpio_tipo,mpio_narea,mpio_nano,shape_Leng,shape_Area,codigo_municipio,total_analisis,ph_validos,media_ph,anio,_merge,area_ha,area_intersectada_ha
0,05,001,05001,ANTIOQUIA,MEDELLÍN,1965,MUNICIPIO,374.741484,2025,1.035380,0.030608,05001,NaN,NaN,NaN,NaN,left_only,37474.148450,0
1,05,002,05002,ANTIOQUIA,ABEJORRAL,1814,MUNICIPIO,506.959782,2024,1.158504,0.041384,05002,NaN,NaN,NaN,NaN,left_only,50695.978227,0
2,05,004,05004,ANTIOQUIA,ABRIAQUÍ,1912,MUNICIPIO,296.974188,2025,0.810667,0.024253,05004,NaN,NaN,NaN,NaN,left_only,29697.418845,0
3,05,021,05021,ANTIOQUIA,ALEJANDRÍA,Decreto departamental 304 de 1907,MUNICIPIO,128.856431,2024,0.705200,0.010535,05021,NaN,NaN,NaN,NaN,left_only,12885.643129,0
4,05,030,05030,ANTIOQUIA,AMAGÁ,1912,MUNICIPIO,84.117145,2024,0.445533,0.006867,05030,NaN,NaN,NaN,NaN,left_only,8411.714470,0


(1122, 19)
dpto_ccdgo               object
mpio_ccdgo               object
mpio_cdpmp               object
dpto_cnmbr               object
mpio_cnmbr               object
mpio_crslc               object
mpio_tipo                object
mpio_narea              float64
mpio_nano                 int32
shape_Leng              float64
shape_Area              float64
codigo_municipio         object
total_analisis          float64
ph_validos              float64
media_ph                float64
anio                    float64
_merge                   object
area_ha                 float64
area_intersectada_ha      int64
dtype: object


In [7]:
datos_pl = pl.read_parquet(INPUT)
print(datos_pl.head())
print(datos_pl.shape)
print(datos_pl.schema)

shape: (5, 19)
┌────────────┬────────────┬────────────┬────────────┬───┬──────┬───────────┬───────────┬───────────┐
│ dpto_ccdgo ┆ mpio_ccdgo ┆ mpio_cdpmp ┆ dpto_cnmbr ┆ … ┆ anio ┆ _merge    ┆ area_ha   ┆ area_inte │
│ ---        ┆ ---        ┆ ---        ┆ ---        ┆   ┆ ---  ┆ ---       ┆ ---       ┆ rsectada_ │
│ str        ┆ str        ┆ str        ┆ str        ┆   ┆ f64  ┆ str       ┆ f64       ┆ ha        │
│            ┆            ┆            ┆            ┆   ┆      ┆           ┆           ┆ ---       │
│            ┆            ┆            ┆            ┆   ┆      ┆           ┆           ┆ i64       │
╞════════════╪════════════╪════════════╪════════════╪═══╪══════╪═══════════╪═══════════╪═══════════╡
│ 05         ┆ 001        ┆ 05001      ┆ ANTIOQUIA  ┆ … ┆ null ┆ left_only ┆ 37474.148 ┆ 0         │
│            ┆            ┆            ┆            ┆   ┆      ┆           ┆ 45        ┆           │
│ 05         ┆ 002        ┆ 05002      ┆ ANTIOQUIA  ┆ … ┆ null ┆ left_only ┆

## 1. Contrato común y consulta · 25 minutos

Proyecta en ambos motores las mismas columnas: `id_fila`, `anio`, `codigo_departamento`, `codigo_municipio`, `ph` y `ph_valido`. Mantén todas las filas. Declara tipos equivalentes, cómo representar nulos en CSV y cómo restaurar códigos con ceros iniciales.

Elige un año y departamento presentes. Filtra y agrupa por municipio para obtener total de análisis, cantidad de pH válidos y media del pH válido. Conserva el grupo municipal ausente, si existe, y la media ausente cuando no hay pH válidos. En pandas utiliza `groupby(..., dropna=False)`; en Polars comprueba el tratamiento equivalente de claves nulas.

**Control:** compara claves y conteos exactos; medias con tolerancia absoluta 0.000001 y relativa cero. Ordena antes de comparar y comprueba también dónde aparecen nulos.

In [11]:
# --- PUNTO 1 CORREGIDO: CONTRATO COMÚN Y CONSULTA ---

# 1. Columnas reales basadas en el esquema de analisis_integrados.parquet
COLUMNAS_CONTRATO = ["anio", "dpto_ccdgo", "codigo_municipio", "media_ph", "ph_validos", "total_analisis"]

TIPOS_PANDAS = {
    "anio": "Int64",
    "dpto_ccdgo": "string",
    "codigo_municipio": "string",
    "media_ph": "float64",
    "ph_validos": "Int64",
    "total_analisis": "float64"
}

TIPOS_POLARS = {
    "anio": pl.Int64,
    "dpto_ccdgo": pl.Utf8,
    "codigo_municipio": pl.Utf8,
    "media_ph": pl.Float64,
    "ph_validos": pl.Int64,
    "total_analisis": pl.Float64
}

# 2. Carga y proyección en Pandas
df_pd_proyectado = pd.read_parquet(INPUT, columns=COLUMNAS_CONTRATO, engine="pyarrow")
for col, dtype in TIPOS_PANDAS.items():
    if col in df_pd_proyectado.columns:
        df_pd_proyectado[col] = df_pd_proyectado[col].astype(dtype)

# 3. Carga y proyección en Polars
df_pl_proyectado = pl.read_parquet(INPUT, columns=COLUMNAS_CONTRATO).cast(TIPOS_POLARS)

# 4. Selección de un año y departamento presentes para el filtro
anio_muestra = int(df_pd_proyectado['anio'].dropna().iloc[0])
dpto_muestra = str(df_pd_proyectado['dpto_ccdgo'].dropna().iloc[0])

print(f"Filtrando para Año: {anio_muestra} y Departamento (dpto_ccdgo): {dpto_muestra}")

# 5. Filtrado y agrupación en Pandas
df_pd_filtrado = df_pd_proyectado[
    (df_pd_proyectado['anio'] == anio_muestra) & 
    (df_pd_proyectado['dpto_ccdgo'] == dpto_muestra)
]

resumen_pd = df_pd_filtrado.groupby(['codigo_municipio'], dropna=False).agg(
    total_registros=('total_analisis', 'sum'),
    ph_validos_suma=('ph_validos', 'sum'),
    media_ph_promedio=('media_ph', 'mean')
).reset_index()

# 6. Filtrado y agrupación equivalente en Polars
df_pl_filtrado = df_pl_proyectado.filter(
    (pl.col('anio') == anio_muestra) & 
    (pl.col('dpto_ccdgo') == dpto_muestra)
)

resumen_pl = df_pl_filtrado.group_by('codigo_municipio', maintain_order=True).agg([
    pl.col('total_analisis').sum().alias('total_registros'),
    pl.col('ph_validos').sum().alias('ph_validos_suma'),
    pl.col('media_ph').mean().alias('media_ph_promedio')
]).sort('codigo_municipio')

# Ordenar Pandas para control
resumen_pd = resumen_pd.sort_values('codigo_municipio').reset_index(drop=True)

display("Resumen Pandas:", resumen_pd.head())
display("Resumen Polars:", resumen_pl.head())

Filtrando para Año: 2025 y Departamento (dpto_ccdgo): 05


'Resumen Pandas:'

,codigo_municipio,total_registros,ph_validos_suma,media_ph_promedio
0,05045,1.0,1,5.710000
1,05051,1.0,1,5.590000
2,05107,3.0,3,6.083333
3,05147,4.0,4,6.010000
4,05154,2.0,2,4.945000


'Resumen Polars:'

codigo_municipio,total_registros,ph_validos_suma,media_ph_promedio
str,f64,i64,f64
"""05045""",1.0,1,5.71
"""05051""",1.0,1,5.59
"""05107""",3.0,3,6.083333
"""05147""",4.0,4,6.01
"""05154""",2.0,2,4.945


**Interpretación:**

Equivalencia y Consistencia entre Motores: Tanto el bloque procesado con Pandas como el procesado con Polars arrojan exactamente los mismos resultados numéricos y tabulares para el filtro aplicado (Año 2025, Departamento 05), confirmando que las operaciones de filtrado y agregación se ejecutaron correctamente en ambos entornos.

Preservación del Formato Territorial: Se observa que los códigos de municipio se conservan de manera idéntica con sus ceros iniciales (por ejemplo, "05045", "05051"), lo cual valida el éxito de la tipificación estricta implementada en el contrato común.   

Validación de Métricas Agregadas: Las columnas de conteo (total_registros, ph_validos_suma) y de promedio aritmético (media_ph_promedio) coinciden de forma absoluta en ambas tablas, garantizando la precisión matemática requerida para la comparación de rendimiento.

## 2. Particionamiento manual desde pandas · 35 minutos

Escribe la misma proyección como CSV y Parquet único. Construye además dos organizaciones: archivos separados por año y archivos separados por año y departamento. Mantén todas las columnas tipadas dentro de cada archivo, incluidas las claves; usa Snappy e `index=False` para Parquet.

Usa `groupby(..., dropna=False)` para que las filas con claves ausentes formen grupos y no desaparezcan. Escribe cada grupo con una llamada directa a `to_parquet`. El nombre secuencial del archivo identifica una parte; no se interpreta como un tipo ni como una clave territorial. Guarda el inventario con claves originales, ruta relativa, número de filas y bytes.

Este ejemplo muestra la mecánica por año. Adáptalo también a año y departamento, en otra carpeta. `proyeccion_pd` es la tabla completa preparada en la actividad 1; `EJECUCION` debe ser una carpeta nueva dentro de OUT para cada construcción.

```python
carpeta = Path(EJECUCION) / "por_anio"
carpeta.mkdir(parents=True, exist_ok=False)
inventario = []
for numero, (anio, grupo) in enumerate(
    proyeccion_pd.groupby("anio", dropna=False, sort=True)
):
    archivo = carpeta / f"parte-{numero:04d}.parquet"
    grupo.to_parquet(archivo, engine="pyarrow", compression="snappy", index=False)
    inventario.append({
        "anio": anio,
        "archivo": str(archivo.relative_to(EJECUCION)),
        "filas": len(grupo),
        "bytes": archivo.stat().st_size,
    })
inventario_anio = pd.DataFrame(inventario)
inventario_anio.to_csv(Path(EJECUCION) / "inventario_anio.csv", index=False)
```

Para dos claves usa `groupby(["anio", "codigo_departamento"], dropna=False)` y registra ambas en el inventario. Muestra su código completo. Al releer el inventario, usa `pd.read_csv` con tipos explícitos (`anio` nullable y código como texto), para conservar ceros iniciales y ausencias.

**Control:** relee todos los archivos por ruta, concatena con `pd.concat(..., ignore_index=True)` y compara con la proyección ordenada por `id_fila`. Comprueba unicidad y conservación de IDs, no solo el total. Compara archivos pequeños, cantidad y bytes entre ambos diseños. Dividir archivos no implica procesamiento paralelo.

In [ ]:
# Desarrollo
# Escribe las cuatro representaciones y presenta su inventario de particiones.

**Interpretación:**

_Presenta los controles obtenidos y explica sus límites._

## 3. Lectura y escritura comparables · 35 minutos

Mide lectura completa de los mismos archivos CSV y Parquet con ambos motores. Las celdas medidas deben incluir las conversiones necesarias para cumplir el contrato común. Mide por separado escritura de CSV y Parquet desde los datos equivalentes ya residentes en memoria; no incluyas la carga en ese tiempo. Usa el mismo orden de columnas, separador, nulos y compresión, y documenta opciones distintas que no puedas igualar.

Usa `time.perf_counter()` inmediatamente antes y después de cada operación. Realiza una ejecución inicial separada y cinco repeticiones, alternando el orden de motores. Escribe cada repetición en una ruta nueva; no acumules archivos dentro de un dataset particionado ya existente. Deja impresión, validación e inventario fuera del intervalo medido.

**Control:** relee los resultados fuera de la medición y verifica equivalencia. Conserva tiempos individuales y mediana. Documenta versiones, CPU, RAM disponible y ubicación de archivos. No describas una lectura como fría sin haber controlado la caché del sistema.

In [ ]:
# Desarrollo
# Mide con llamadas directas a read_csv/read_parquet y to_csv/to_parquet o write_csv/write_parquet.

**Interpretación:**

_Presenta los controles obtenidos y explica sus límites._

## 4. Procesamiento y consulta desde disco · 40 minutos

Separa dos experimentos:

1. **En memoria:** mide solo el filtro y la agregación sobre tablas ya cargadas en pandas y Polars.
2. **Desde disco:** mide hasta obtener el resultado del filtro y la agregación. Compara archivo único, archivos por año y archivos por año/departamento.

Para las partes, lee el inventario CSV dentro del intervalo medido, filtra sus claves con pandas y obtiene la lista de rutas. Para un año solo, selecciona sus partes y aplica el departamento sobre los registros leídos; para año/departamento, selecciona ambas claves. Usa exactamente la misma lista de archivos para los dos motores. Registra qué rutas seleccionaste. La tabla de inventario es pequeña y se usa igual en ambos recorridos.

En pandas, muestra un bucle que llame a `pd.read_parquet` para cada ruta y luego `pd.concat`. En Polars, muestra un bucle con `pl.read_parquet` y luego `pl.concat`. En cada caso, aplica las mismas condiciones analíticas y agregaciones. El ejemplo siguiente explica la lectura pandas; escribe su equivalente Polars:

```python
partes_pd = []
for ruta in rutas_seleccionadas:
    parte_pd = pd.read_parquet(
        ruta,
        engine="pyarrow",
        columns=["id_fila", "anio", "codigo_departamento",
                 "codigo_municipio", "ph", "ph_valido"]
    )
    partes_pd.append(parte_pd)
seleccion_pd = pd.concat(partes_pd, ignore_index=True)
```

Antes de medir, verifica que la consulta elegida tiene archivos candidatos. Si no los tiene, presenta un resultado vacío con esquema explícito; no llames `concat` sobre una lista vacía.

**Consulta diferida adicional:** sobre el Parquet único usa `pl.scan_parquet`, selección de columnas, filtro, agregación y `collect()`. Mide la ejecución completa y muestra `explain()` fuera del tiempo medido. Contrasta esta variante con `pl.read_parquet`; así separas el efecto de la consulta diferida del de dividir archivos.

**Control:** cinco repeticiones por variante, mismos datos y resultados materializados. Incluye lectura del inventario, selección de rutas, lectura de partes, concatenación, filtro y agregación en el experimento desde disco. Excluye impresión y comprobaciones. Los bytes de archivos seleccionados no son una medida de bytes físicos leídos. No compares construir un plan con ejecutarlo.

In [ ]:
# Desarrollo
# Ejecuta los dos experimentos y comprueba resultados materializados equivalentes.

**Interpretación:**

_Presenta los controles obtenidos y explica sus límites._

## 5. Llevar los resultados tabulares a un mapa · 30 minutos

Lee directamente la capa `municipios` del GeoPackage de PD04. Muestra columnas, CRS y cantidad de geometrías. Selecciona únicamente código y geometría para evitar reutilizar por accidente los indicadores del año anterior.

Convierte el resultado agregado de Polars con `.to_pandas()` después de medir. Une por separado el resultado pandas y el resultado Polars con los mismos polígonos mediante `merge(..., how="left", validate="one_to_one")`. Conserva el código como texto y verifica antes la unicidad. Presenta resultados sin geometría y polígonos sin resultado.

Genera dos mapas del departamento y año elegidos, uno por motor, usando igual extensión, columna, límites de color, leyenda y tratamiento de ausentes. Guarda `comparacion_mapas.png`. Comprueba numéricamente las claves, conteos y medias antes de juzgar la igualdad visual.

**Control:** las geometrías no participan en el benchmark tabular ni se convierten a texto para medirlas con Polars. La unión y la representación son trabajo de GeoPandas fuera de esos tiempos. Un mapa igual no sustituye una comparación de valores. La cantidad de registros no demuestra cobertura completa de suelos.

In [ ]:
geometrias = gpd.read_file(
    REPO + "/kit/salidas/soluciones/PD04/resultados_espaciales.gpkg",
    layer="municipios",
    engine="pyogrio"
)
display(geometrias.head())
print(geometrias.shape, geometrias.crs)

In [ ]:
# Desarrollo
# Une ambos resultados a las mismas geometrías y comprueba tablas y mapas.

**Interpretación:**

_Presenta los controles y explica el alcance del resultado._

## 6. Reejecución, evidencia y decisión · 45 minutos

Repite la construcción de los archivos en otra carpeta de ejecución nueva. No borres carpetas de datos originales ni reutilices un directorio particionado con archivos viejos. Comprueba contenido y total de registros; el nombre de los archivos o su hash no tiene por qué coincidir para que el contenido sea equivalente.

Guarda `mediciones.csv` con motor, versión, operación, formato, particionamiento, repetición, segundos, filas de entrada y filas de salida. Guarda aparte la ejecución inicial; calcula la mediana solo con las cinco repeticiones medidas. Añade inventario de bytes/archivos y controles de equivalencia.

Responde: ¿qué parte mejora con Parquet?, ¿cuándo ayuda la selección de particiones?, ¿qué diseño genera archivos demasiado pequeños?, ¿qué cambia entre tiempo en memoria y tiempo desde disco?, ¿qué elegirías para este conjunto y por qué?

**Control:** no presupongas que Polars o el particionamiento ganarán. Diferencias pequeñas o variables deben describirse como tales. El tamaño de un DataFrame no equivale a la memoria máxima del proceso.

In [ ]:
# Desarrollo
# Concilia la segunda ejecución, exporta las mediciones y redacta la recomendación.

**Interpretación:**

_Presenta los controles obtenidos y explica sus límites._

## Entrega y referencias de API

`soluciones/PD05_particionamiento_pandas_polars.ipynb`, mediciones, inventarios, comparación cartográfica y controles regenerables en `kit/salidas/soluciones/PD05/`. Incluye una recomendación basada en resultados; no se califica obtener el menor tiempo.

- [pandas: escritura de archivos Parquet](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.to_parquet.html)
- [Polars: fuentes y ejecución diferida](https://docs.pola.rs/user-guide/lazy/sources_sinks/)
- [Polars: scan_parquet](https://docs.pola.rs/api/python/stable/reference/api/polars.scan_parquet.html)

Usa las versiones fijadas por el curso; si la documentación más reciente muestra una opción diferente, comprueba su disponibilidad antes de usarla.